<a href="https://colab.research.google.com/github/MysteriousCode786/Customer-Churn-Sentiment-Analytics/blob/main/Customer_Churn_%26_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Set seed for reproducibility of random numbers
np.random.seed(42)

# Define the number of customers for our dataset
num_customers = 500

# Generate Numerical Behavioural Data
customer_ids = [f"CUST_{i:04d}" for i in range(1, num_customers + 1)]
age = np.random.randint(18, 65, size = num_customers)
tenure_months = np.random.randint(1, 24, size = num_customers)
monthly_logins = np.random.randint(1, 30, size = num_customers)
support_tickets = np.random.randint(0, 5, size = num_customers)

# Generate text Reviews & Corresponding Churn Indicator
# We create a logic where bad reviews and high support tickets correlate with Churn
reviews = []
churn_labels = []

positive_reviews = [
    "Amazing app! Extremely smooth user interface and very helpful customer service.",
    "Highly recommended. It saves me so much time every single day.",
    "Love the new updates, works perfectly fine on my device.",
    "Great experience so far. The features are intuitive and easy to use.",
]

negative_reviews = [
    "Terrible experience. The app crashes constantly during checkout.",
    "Very frustrated with the frequent bugs and slow loading times.",
    "Customer support is useless. Worst service ever experienced.",
    "The recent update ruined everything. Planning to cancel my subscription.",
]

for i in range(num_customers):
  # If a customer has high support tickets, they are more likely to leave(churn) and leave a bad review.
  if support_tickets[i] >= 3 or monthly_logins[i] < 5:
    reviews.append(np.random.choice(negative_reviews))
    churn_labels.append(1)  # 1 means churned (left the platform)
  else:
    # Customers with good usage metrics are likely to stay and leave positive feedback
    reviews.append(np.random.choice(positive_reviews))
    churn_labels.append(0)  # 0 means Retained (Staying)

# Combine everything into a structured Pandas DataFrame
data = pd.DataFrame(
    {
        "CustomerID": customer_ids,
        "Age": age,
        "TenureMonths": tenure_months,
        "MonthlyLogins": monthly_logins,
        "SupportTickets": support_tickets,
        "CustomerReview": reviews,
        "Churn": churn_labels,
    }
)

In [ ]:
print("Dataset successfully generated! Here is a preview.")
print(data.head())

Dataset successfully generated! Here is a preview.
  CustomerID  Age  TenureMonths  MonthlyLogins  SupportTickets  \
0  CUST_0001   56             1             11               4   
1  CUST_0002   46            19             15               2   
2  CUST_0003   32            11             24               1   
3  CUST_0004   60             5              6               0   
4  CUST_0005   25            12             23               2   

                                      CustomerReview  Churn  
0  Customer support is useless. Worst service eve...      1  
1  Highly recommended. It saves me so much time e...      0  
2  Amazing app! Extremely smooth user interface a...      0  
3  Love the new updates, works perfectly fine on ...      0  
4  Amazing app! Extremely smooth user interface a...      0  


In [ ]:
# Save the generated dataset to a CSV file
data.to_csv("customer_churn_data.csv", index = False)

In [ ]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download the VADER lexicon (vocabular dictionary package needed for sentiment analysis)
nltk.download("vader_lexicon")

# Load the dataset (customer_churn_data.csv)
df = pd.read_csv("customer_churn_data.csv")

# Initialize the VADER Sentiment Intensity Analyzer
sia = SentimentIntensityAnalyzer()

# Define a function to calculate only the compound sentiment score
def get_sentiment_score(text):
  # polarity_scores return a dictionary with pos, neg, neu, and compound scores
  scores = sia.polarity_scores(text)
  return scores['compound']

# Apply the function to the CustomerReview column and create a new feature
df["SentimentScore"] = df['CustomerReview'].apply(get_sentiment_score)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:
# Preview the dataset with the new SentimentScore column
print("Sentiment Analysis completed successfully! Previewing new column:")
print(df[["CustomerReview", "SentimentScore", "Churn"]].head())

Sentiment Analysis completed successfully! Previewing new column:
                                      CustomerReview  SentimentScore  Churn
0  Customer support is useless. Worst service eve...         -0.6369      1
1  Highly recommended. It saves me so much time e...          0.2716      0
2  Amazing app! Extremely smooth user interface a...          0.8012      0
3  Love the new updates, works perfectly fine on ...          0.8807      0
4  Amazing app! Extremely smooth user interface a...          0.8012      0


In [ ]:
# Save the updated DataFrame
df.to_csv("customer_churn_with_sentiment.csv", index = False)

In [ ]:
# Load the dataset (customer_churn_with_sentiment.csv)
df = pd.read_csv("customer_churn_with_sentiment.csv")

# Check basic summary statistics of dataset
print("--- Basic Data Summary ---")
print(df.describe())
print("\n" + "=" * 50 + "\n")

# Analyze the realtionaship between Sentiment Score and Churn
# We group by 'Churn and calculate the average Sentiment Score for each group
print("--- Average Sentiment Score by Churn Status ---")
sentiment_trend = df.groupby("Churn")["SentimentScore"].mean()
print(sentiment_trend)
print("\n" + "=" * 50 + "\n")

# Analyze how Support Tickets impact Churn rate
print("--- Average Support Tickets by Churn Status ---")
ticket_trend = df.groupby("Churn")["SupportTickets"].mean()
print(ticket_trend)

--- Basic Data Summary ---
              Age  TenureMonths  MonthlyLogins  SupportTickets       Churn  \
count  500.000000    500.000000     500.000000      500.000000  500.000000   
mean    41.278000     11.896000      14.860000        2.018000    0.484000   
std     13.389072      6.683001       8.568737        1.411971    0.500244   
min     18.000000      1.000000       1.000000        0.000000    0.000000   
25%     30.000000      6.000000       7.750000        1.000000    0.000000   
50%     42.000000     12.000000      15.000000        2.000000    0.000000   
75%     52.000000     18.000000      22.000000        3.000000    1.000000   
max     64.000000     23.000000      29.000000        4.000000    1.000000   

       SentimentScore  
count      500.000000  
mean         0.069283  
std          0.651927  
min         -0.636900  
25%         -0.570900  
50%          0.271600  
75%          0.801200  
max          0.880700  


--- Average Sentiment Score by Churn Status ---
Chur

In [ ]:
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Separate Features (X) and Target Variable (y)
# We drop columns that are text-based or just unique IDs
X = df.drop(columns = ["CustomerID", "CustomerReview", "Churn"])
y = df["Churn"]

# Split data into Training set (80%) and Testing set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

# Initialize and Train the Random Forest Classifier Model
# random_state ensures the tree splits are reproducible
model = RandomForestClassifier(random_state = 42, n_estimators = 100)
model.fit(X_train, y_train)

# make predicitons on the Test set to check performance
y_pred = model.predict(X_test)

# Evaluate the model performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

# Save the trained model and the columns list
with open("churn_model.pkl", "wb") as model_file:
  pickle.dump(model, model_file)


Model Accuracy: 100.00%

--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        49
           1       1.00      1.00      1.00        51

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

